<!-- # Convert NEUROVASC into MEDS-OWL -->

## Convert NEUROVASC 2.0 into MEDS-KG

In [ ]:
import os

N2_ETL_OUTPUT = "MEDS_cohort"
N2_ETL_INTERMEDIATE = "pre_MEDS"
N2_ETL_INPUT = "raw_input"

EXPORT_DIR = "exports"
ETL_LABELS = f"{EXPORT_DIR}/labels"

os.makedirs(N2_ETL_INPUT, exist_ok=True)
os.makedirs(N2_ETL_INTERMEDIATE, exist_ok=True)
os.makedirs(N2_ETL_OUTPUT, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)
os.makedirs(ETL_LABELS, exist_ok=True)

TIME_OPT = "TS"

In [ ]:
import polars as pl
from utils.neurovasc_meta import SCHEMA_OVERRIDES

# df_input = pl.read_csv(f"{N2_ETL_INPUT}/synthetic_data_sdv.csv")
df_input = pl.read_csv(
    f"{N2_ETL_INPUT}/data.csv",
    # ignore_errors=True,
    schema_overrides=SCHEMA_OVERRIDES,
    try_parse_dates=True,
)
df_input = df_input.filter(pl.col("Outcome").is_not_null())

In [ ]:
from utils.pre_MEDS import generate_patient_timestamps
from utils.pre_MEDS import rebalance_synth

df_input = generate_patient_timestamps(df_input)  # only for synthetic dataset
# df_input = rebalance_synth(df_input)  # only for synthetic

In [ ]:
NUM_PATIENTS = len(set(df_input["Patient_ID"]))

In [ ]:
from utils.pre_MEDS import generate_meds_preprocessed
from shutil import copy

meds_path = f"{ETL_LABELS}/outcomes_meds_{TIME_OPT}_{NUM_PATIENTS}.joblib"

(contextual, sequential, outcomes) = generate_meds_preprocessed(
    df_input,
    output_path=N2_ETL_INTERMEDIATE,
    outcome_path=f"{ETL_LABELS}/outcomes_meds_{TIME_OPT}_{NUM_PATIENTS}.joblib",
)

sphn_path = meds_path.replace("meds", "sphn_pc")
copy(meds_path, sphn_path)

In [ ]:
from MEDS_transforms.runner import main
import shutil

shutil.rmtree(N2_ETL_OUTPUT)

main(
    [
        "pkg://MEDS_extract.configs._extract.yaml",
        "--overrides",
        f"input_dir={N2_ETL_INTERMEDIATE}",
        f"output_dir={N2_ETL_OUTPUT}",
        "event_conversion_config_fp=MESSY.yaml",
        "dataset.name=Neurovasc_v2",
        "dataset.version=2.0",
    ]
)

In [ ]:
cols = [
    "Weight",
    "Number_of_Visited_Departments",
    "Length_of_Stay",
    "Glasgow_Coma_Scale",
    "Fisher_Score",
    "WFNS_Score",
]


def filter_nulls(dt: pl.DataFrame):
    return (
        dt.filter(
            ~(pl.col("code").str.contains_any(cols) & pl.col("numeric_value").is_null())
        )
        # .filter(~pl.col("code").str.contains("UNK"))
    )


for p in ["train", "held_out", "tuning"]:
    pp = f"{N2_ETL_OUTPUT}/data/{p}/0.parquet"
    filter_nulls(pl.read_parquet(pp)).write_parquet(pp)

In [ ]:
# Number of events
print(len(pl.read_parquet(f"{N2_ETL_OUTPUT}/data/**/0.parquet")))

# CLASS BALANCE
df_input.group_by("Patient_ID").first().group_by("Outcome").len().with_columns(
    (pl.col("len") / pl.col("len").sum() * 100).alias("percent")
)

In [ ]:
import shap
import joblib
from pathlib import Path
import numpy as np
import pandas as pd

from rdflib import Namespace
import json
import re

CLASSES = ["BackHome", "Rehab", "Death"]

def save_best_features(output_dir, explainer, feature_names, X):
    import shap
    import matplotlib.pyplot as plt

    shap_values = explainer.shap_values(X)

    for class_idx, class_name in enumerate(CLASSES):
        shap.summary_plot(
            shap_values[:, :, class_idx],
            X,
            feature_names=feature_names,
            show=False,
        )

        plt.savefig(
            f"{output_dir}/shap_class_{class_name}.png",  # type: ignore
            bbox_inches="tight",
            dpi=300,
        )
        plt.close()

def save_top_shap_features_per_class(
    explainer,
    output_dir,
    X,
    feature_names,
    classes,
    quantile=0.95, # Keep all features whose importance is in the top 5% of values
):
    """
    Select SHAP features based on quantile threshold instead of top-k.
    """

    shap_values = explainer.shap_values(X)

    feature_names = np.array(
        [f.replace("_count", "").replace("//", "_") for f in feature_names]
    )

    top_features_per_class = {}
    rows = []

    for class_idx, class_name in enumerate(classes):
        class_shap = shap_values[:, :, class_idx]

        # importance per feature
        importance = np.abs(class_shap).mean(axis=0)

        # quantile threshold
        threshold = np.quantile(importance, quantile)

        selected_idx = np.where(importance >= threshold)[0]

        sorted_idx = selected_idx[np.argsort(importance[selected_idx])[::-1]]

        sorted_features = feature_names[sorted_idx]
        sorted_importance = importance[sorted_idx]

        top_features_per_class[class_name] = {
            "features": sorted_features,
            "importance": sorted_importance,
            "threshold": threshold,
        }

        for rank, (f, v) in enumerate(
            zip(sorted_features, sorted_importance),
            start=1,
        ):
            rows.append({
                "class": class_name,
                "rank": rank,
                "feature": f,
                "importance": v,
                "threshold": threshold,
            })

    df = pd.DataFrame(rows)

    output_csv = f"{output_dir}/shap_features_quantile_{quantile}.csv"
    df.to_csv(output_csv, index=False)
    np.save(f"{output_dir}/features_{quantile}.npy", df["feature"].unique())

    return top_features_per_class

NS_DATA = Namespace("https://teamheka.github.io/meds-data/")
NS_ONTO = Namespace("https://teamheka.github.io/meds-ontology#")
NS_CODE = Namespace(f"{NS_DATA}code/")


def sanitize_for_uri(feature_name: str) -> str:
    s = " ".join(
        feature_name.replace("\\", "\\\\")
        .replace("\r\n", "\n")
        .replace("\t", "\n")
        .replace("\r", "\n")
        .replace('"', "")
        .split()
    )

    unsafe_chars = r'[<>"{}|\\^`\[\]\s]'

    s = re.sub(unsafe_chars, " ", s)
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"_+", "_", s)
    s = s.strip("_")
    return s


MODEL = "xgboost"
QUANTILE  = 0.95
DATASET = "exports"

EXPORT_DIR = f"{DATASET}/metrics_503"

feature_names = joblib.load(f"{EXPORT_DIR}/feature_names.joblib")
X = joblib.load(f"{EXPORT_DIR}/X.joblib")

model_dir = Path(f"{EXPORT_DIR}/{MODEL}/models")

best_model_path = str(
    max(
        model_dir.glob(f"{MODEL}_best_fold*_auc_*.joblib"),
        key=lambda p: float(p.stem.split("_auc_")[-1]),
    )
)

explainer = shap.TreeExplainer(model=joblib.load(best_model_path))

save_best_features(
    output_dir=f"{EXPORT_DIR}/{MODEL}",
    explainer=explainer,
    feature_names=feature_names,
    X=X,
)

top_features_per_class = save_top_shap_features_per_class(
    explainer=explainer,
    X=X,
    feature_names=feature_names,
    classes=CLASSES,
    output_dir=f"{EXPORT_DIR}/{MODEL}",
    quantile=QUANTILE
)


all_features = []
for class_name, content in top_features_per_class.items():
    all_features.extend(content["features"])

_dict = {
    NS_CODE[sanitize_for_uri(feature_name)]: NS_ONTO[
        f"has{sanitize_for_uri(feature_name)}"
    ]
    for feature_name in set(all_features)
}

with open(f"{EXPORT_DIR}/{MODEL}/onto_features_dict_{QUANTILE}.json", "w") as f:
    json.dump(_dict, f, indent=2)

In [ ]:
import os
import polars as pl


def init_dirs(root = "exports"):
    export_dir = f"{root}"
    outcomes_dir = f"{export_dir}/labels"
    meds_cohort_dir = f"{export_dir}/meds/MEDS_cohort"
    os.makedirs(export_dir, exist_ok=True)
    os.makedirs(outcomes_dir, exist_ok=True)
    os.makedirs(meds_cohort_dir, exist_ok=True)
    os.makedirs(f"{export_dir}/meds/MEDS_cohort/metadata", exist_ok=True)

    return export_dir, outcomes_dir, meds_cohort_dir

### 3.4) Filter by best features

In [ ]:
import numpy as np
import re

from utils.processing import create_meds_cohort

def clean_xgb_feature_name(n):
    n = str(n)
    n = re.sub(r"[\[\]<>]", "", n)  # remove forbidden chars
    #n = re.sub(r"[^0-9a-zA-Z_]+", "_", n)  # replace other specials
    # n = re.sub(r"_+", "_", n)              # collapse multiple _
    n = n.replace("//", "_")
    n = n.strip("_")
    return n

QUANTILE = 0.95
SOURCE = "exports"
OUTPUT = f"exports-{QUANTILE}"

arr = np.load(f"{SOURCE}/metrics_503/xgboost/features_{QUANTILE}.npy", allow_pickle=True)
export_dir, outcomes_dir, meds_cohort_dir = init_dirs(root=OUTPUT)

events = pl.scan_parquet("MEDS_cohort/data/**/*.parquet", low_memory=True).collect(engine="streaming")
print(len(events))

events = events.filter(
    pl.col("code").map_elements(clean_xgb_feature_name, return_dtype=pl.Utf8).is_in(arr)
)
print(len(events))

create_meds_cohort(
    events,
    orig_dir="MEDS_cohort",
    output_dir=meds_cohort_dir,
    columns=["subject_id", "code", "time", "numeric_value"],
)


In [ ]:
from meds2rdf import MedsRDFConverter
from meds2rdf.sinks import NTriplesSink
from meds2rdf.config import Config, MEDSSchema
from pathlib import Path

engine = MedsRDFConverter(f"exports-0.95/meds/MEDS_cohort")

graph_dir = Path("exports-0.95") / f"meds_503"

engine.convert(
    sink=NTriplesSink(graph_dir, gzip_mode=False),
    cfg=Config(schemas={MEDSSchema.CODES}),
)

### b) Predict patient outcomes with tabular-based models

In [ ]:
from utils.tabular import run_tabulars_models

CLASSES = ["BackHome", "Rehab", "Death"]

X, y = run_tabulars_models(
    meds_root=N2_ETL_OUTPUT,
    classes=CLASSES,
    outcomes_path=f"{ETL_LABELS}/outcomes_meds_{TIME_OPT}_{NUM_PATIENTS}.joblib",
    result_dir=f"{EXPORT_DIR}/metrics_{NUM_PATIENTS}",
    save_model=True,
)

feature_names = X.columns.to_list()  # type: ignore

joblib.dump(
    X.columns.to_list(),
    f"{EXPORT_DIR}/metrics_{NUM_PATIENTS}/feature_names.joblib",
)
joblib.dump(X, f"{EXPORT_DIR}/metrics_{NUM_PATIENTS}/X.joblib")

### c) Explain most important features through SHAP

In [ ]:
import shap
import joblib
from pathlib import Path

model_name = "xgboost"

model_dir = Path(f"{EXPORT_DIR}/metrics_{NUM_PATIENTS}/{model_name}/models")

best_model_path = str(
    max(
        model_dir.glob(f"{model_name}_best_fold*_auc_*.joblib"),
        key=lambda p: float(p.stem.split("_auc_")[-1]),
    )
)

explainer = shap.TreeExplainer(model=joblib.load(best_model_path))

In [ ]:
import shap
import matplotlib.pyplot as plt

shap_values = explainer.shap_values(X)

for class_idx, class_name in enumerate(CLASSES):
    shap.summary_plot(
        shap_values[:, :, class_idx],
        X,
        feature_names=feature_names,
        show=False,
    )

    plt.savefig(
        f"{EXPORT_DIR}/metrics_{NUM_PATIENTS}/{model_name}/shap_class_{class_name}.png",  # type: ignore
        bbox_inches="tight",
        dpi=300,
    )
    plt.close()

In [ ]:
sample_idx = 0

shap_exp = explainer(X)

waterfall_exp = shap.Explanation(
    values=shap_exp.values[sample_idx, :, class_idx],  # type: ignore
    base_values=shap_exp.base_values[sample_idx, class_idx],  # type: ignore
    data=X.iloc[sample_idx],  # type: ignore
    feature_names=feature_names,
)

shap.plots.waterfall(
    waterfall_exp,
    max_display=20,
    show=False,
)

plt.savefig(
    f"{EXPORT_DIR}/metrics_{NUM_PATIENTS}/{model_name}/shap_waterfall_class_{CLASSES[y[sample_idx]]}.png",
    bbox_inches="tight",
    dpi=300,
)

plt.close()

In [ ]:
force_plot = shap.force_plot(
    base_value=shap_exp.base_values[sample_idx, class_idx],  # type: ignore
    shap_values=shap_exp.values[sample_idx, :, class_idx],  # type: ignore
    features=X.iloc[sample_idx],
    feature_names=feature_names,
    matplotlib=True,
    show=False,
)

plt.savefig(
    f"{EXPORT_DIR}/metrics_{NUM_PATIENTS}/{model_name}/shap_force_class_{CLASSES[y[sample_idx]]}.png",
    bbox_inches="tight",
    dpi=300,
)

plt.close()

### d) Convert MIMIC into SPHN-RDF

In [ ]:
import os

ROOT = "MEDS_cohort"
EXPORT_FOLDER = "exports"

df = (
    pl.scan_parquet(f"{ROOT}/data/**/0.parquet")
    .select(["subject_id", "code", "time", "numeric_value"])
    .collect(engine="streaming")
)

NUM_PATIENTS = len(df.group_by(pl.col("subject_id")).len())
OUTPUT_FOLDER = f"{EXPORT_FOLDER}/sphn_pc_{NUM_PATIENTS}"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [ ]:
from tqdm import tqdm

from utils.sphn_mapping import NTBatchWriter, build_rdf_event

ROWS_PER_FILE = 100_000

writer = NTBatchWriter(OUTPUT_FOLDER, ROWS_PER_FILE)

for row in tqdm(df.iter_rows(), total=len(df), desc="Generating RDF"):
    rdf = build_rdf_event(row)
    if rdf is None:
        continue

    writer.write(rdf)

writer.close()